In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera


df = pd.read_excel("/content/USA CP2.xlsx")

df['GDP_diff'] = df['GDP'].diff()
df['CO2_diff'] = df['CO2'].diff()
df['Credit_diff'] = df['Credit'].diff()

df_var = df[['GDP_diff', 'CO2_diff', 'Credit_diff']].dropna()

# Базовая модель VAR(1)

In [ ]:
model1 = VAR(df_var)
model1_fitted = model1.fit(1)
print(model1_fitted.summary())

In [ ]:
print("=== Анализ остатков: model1_fitted (VAR(1)) ===")

# Durbin-Watson
print("\nDurbin-Watson:")
for col, val in zip(df_var.columns, durbin_watson(model1_fitted.resid)):
    print(f"{col}: {val:.2f}")

# Ljung-Box по каждому остатку
print("\nLjung-Box (p-value по каждому остатку):")
for name in df_var.columns:
    lb = acorr_ljungbox(model1_fitted.resid[name], lags=[1], return_df=True)
    print(f"{name}: p-value = {lb['lb_pvalue'].values[0]:.3f}")

# Jarque-Bera
print("\nJarque-Bera (нормальность):")
for name in df_var.columns:
    jb = jarque_bera(model1_fitted.resid[name])
    print(f"{name}: JB={jb[0]:.2f}, p-value={jb[1]:.3f}")

# Альтернативная модель VAR(2)

In [ ]:
model2_fitted = model1.fit(2)
print(model2_fitted.summary())

In [ ]:
print("=== Анализ остатков: model2_fitted (VAR(2)) ===")

print("\nDurbin-Watson:")
for col, val in zip(df_var.columns, durbin_watson(model2_fitted.resid)):
    print(f"{col}: {val:.2f}")

print("\nLjung-Box (p-value по каждому остатку):")
for name in df_var.columns:
    lb = acorr_ljungbox(model2_fitted.resid[name], lags=[1], return_df=True)
    print(f"{name}: p-value = {lb['lb_pvalue'].values[0]:.3f}")

print("\nJarque-Bera (нормальность):")
for name in df_var.columns:
    jb = jarque_bera(model2_fitted.resid[name])
    print(f"{name}: JB={jb[0]:.2f}, p-value={jb[1]:.3f}")

# VAR(1) + тренд

In [ ]:
model3_fitted = model1.fit(1, trend='ct')  # c = constant, t = trend
print(model3_fitted.summary())

In [ ]:
print("=== Анализ остатков: model3_fitted (VAR(1) + тренд) ===")

print("\nDurbin-Watson:")
for col, val in zip(df_var.columns, durbin_watson(model3_fitted.resid)):
    print(f"{col}: {val:.2f}")

print("\nLjung-Box (p-value по каждому остатку):")
for name in df_var.columns:
    lb = acorr_ljungbox(model3_fitted.resid[name], lags=[1], return_df=True)
    print(f"{name}: p-value = {lb['lb_pvalue'].values[0]:.3f}")

print("\nJarque-Bera (нормальность):")
for name in df_var.columns:
    jb = jarque_bera(model3_fitted.resid[name])
    print(f"{name}: JB={jb[0]:.2f}, p-value={jb[1]:.3f}")

# Моделирование: оценка альтернативных спецификаций VAR

В данном разделе оценивались различные спецификации моделей VAR на первых разностях переменных GDP, CO₂ и Credit. Основная цель — выбрать наиболее адекватную модель для краткосрочного анализа взаимосвязей между переменными.

В рамках задания были построены и протестированы три спецификации:

VAR(1) — базовая модель с одним лагом;

VAR(2) — модель с двумя лагами;

VAR(1) с трендом — включает линейный временной тренд.

| № | Спецификация модели | AIC      | BIC      | Стационарность\*    | Автокорреляция остатков (DW / Ljung–Box) | Нормальность остатков (JB-test) | Вывод                    |
| - | ------------------- | -------- | -------- | ------------------- | ---------------------------------------- | ------------------------------- | ------------------------ |
| 1 | VAR(1)              | **1.06** | **1.57** | ΔGDP, ΔCO₂, ΔCredit | DW ≈ 2, p > 0.35 — нет                   | p > 0.05 (кроме Credit)         | ЛУЧШАЯ МОДЕЛЬ          |
| 2 | VAR(2)              | 1.14     | 2.05     | ΔGDP, ΔCO₂, ΔCredit | DW ≈ 2, p > 0.60 — нет                   | Все p > 0.1                     | Почти равная по качеству |
| 3 | VAR(1) + тренд      | 1.20     | 1.83     | ΔGDP, ΔCO₂, ΔCredit | DW ≈ 2, p > 0.35 — нет                   | Credit: p = 0.000 ❌             | Хуже по критериям        |


* Стационарность обеспечивается за счёт первых разностей.

# **Анализ остатков**

Durbin–Watson: все модели показывают значения в пределах 1.7–2.0 → проблем автокорреляции нет.

Ljung–Box: p-value > 0.35 во всех моделях → остатки не автокоррелированы.

Jarque–Bera: только в model1_fitted и model2_fitted все остатки (почти) нормальны.
В model3_fitted (с трендом) остатки по Credit_diff — сильно не нормальны (p = 0.000).

**Наилучшей спецификацией признана VAR(1) без тренда, так как:** имеет наименьшее значение AIC и BIC, все остатки — почти нормально распределены, нет автокорреляции, модель стабильна и экономически интерпретируема.

**Математическая форма VAR(1):**

Модель VAR(1) включает три уравнения, каждое из которых описывает поведение одной переменной на основе её собственных лагов и лагов других переменных:

ΔGDPₜ = α₁ + β₁·ΔGDPₜ₋₁ + β₂·ΔCO₂ₜ₋₁ + β₃·ΔCreditₜ₋₁ + ε₁ₜ

ΔCO₂ₜ = α₂ + γ₁·ΔGDPₜ₋₁ + γ₂·ΔCO₂ₜ₋₁ + γ₃·ΔCreditₜ₋₁ + ε₂ₜ

ΔCreditₜ = α₃ + δ₁·ΔGDPₜ₋₁ + δ₂·ΔCO₂ₜ₋₁ + δ₃·ΔCreditₜ₋₁ + ε₃ₜ

# ИТОГ

VAR(1) без тренда показала наилучшие характеристики по информационным критериям и остаточным свойствам. Другие спецификации, включая более длинные лаги и тренд, либо уступают по качеству, либо не улучшают интерпретируемость модели.

# 7

| Какая модель оценивалась? | Интерпретация                                                                                                 | Коинтеграционное соотношение, ECM, ADL               | Долгосрочные и краткосрочные эффекты, период возврата к равновесию                        |
| ------------------------- | ------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| **VAR(1)**                | Анализирует краткосрочные взаимосвязи между GDP, CO₂ и Credit. Использует лаговые значения каждой переменной. | Отсутствует (тест Джохансена не выявил коинтеграции) | Выявлены краткосрочные эффекты: Credit → GDP и CO₂. Шоки постепенно затухают за 3–4 года. |


В рамках данного исследования была оценена модель векторной авторегрессии (VAR(1)) на первых разностях переменных: ВВП на душу населения (GDP), выбросы CO₂ на душу населения (CO₂), объём кредитования частного сектора (Credit, % от ВВП).

Поскольку тест Джохансена не выявил коинтеграции между переменными, модели типа VECM или ECM не применялись. VAR-модель позволила проанализировать краткосрочные взаимосвязи между переменными, включая направленные шоки и динамику реакции экономических индикаторов.

Анализ функций импульсного отклика (IRF) позволил выявить характер краткосрочной реакции экономических переменных на внезапные изменения (шоки) в системе. Наиболее сильную динамику продемонстрировали переменные GDP и CO₂ в ответ на шок в объёме кредитования: увеличение кредитов сопровождалось ростом как ВВП, так и выбросов углекислого газа. Наиболее выраженный эффект наблюдался в первые 2–3 года, после чего отклик постепенно затухал. Это подтверждает наличие причинно-следственной связи между финансированием и экономической активностью, а также подчёркивает, что финансовый сектор способен усиливать нагрузку на экологию. При этом влияние шоков в GDP и CO₂ на другие переменные оказалось менее стабильным и слабее выраженным.

Дополнительно была проведена декомпозиция дисперсии ошибки прогноза (FEVD), чтобы выяснить, какие факторы влияют на долгосрочную изменчивость показателей. Выяснилось, что основная часть колебаний ВВП на начальных горизонтах объясняется его собственными значениями, однако с течением времени растёт доля влияния кредитования. Для CO₂ ситуация схожа: в краткосрочном периоде доминируют собственные лаги, но в долгосрочной перспективе вклад финансового сектора становится заметным. Что касается самого кредитования, его изменчивость частично объясняется ростом ВВП, что логично: в периоды экономического подъёма наблюдается увеличение кредитной активности.

Таким образом, модель VAR(1), построенная на первых разностях, выявила устойчивые краткосрочные взаимосвязи между экономическим ростом, уровнем кредитования и выбросами CO₂. Несмотря на отсутствие долгосрочного равновесия (коинтеграции), обнаруженные эффекты важны для понимания текущей макроэкономической динамики. Полученные результаты подтверждают основные выводы статьи Abbasi & Riaz (2016), но уже в контексте США: кредитование действительно связано с ростом экономики и одновременно сопровождается увеличением экологических рисков. Это подчёркивает необходимость сбалансированной финансовой политики, ориентированной не только на экономический рост, но и на устойчивое развитие.